# MINERVA — Pandemic & Vaccine Diplomacy Simulator

A LangGraph multi-agent environment implementing *The Refined Environmental Engine*.

**Core loop (1 round = 10 days, 10 rounds total):**

| # | Node | What happens |
|---|------|--------------|
| 1 | `core_engine`   | Round init, ledger snapshot |
| 2 | `pandemic`      | Containment roll vs `alpha` → **alpha-scaled 10% infection wave** |
| 3 | `production`    | `R = R_max · (P/P_initial)`, capped by capital (1 capital = 1 vaccine) |
| 4 | `ranking_agent` | Deterministic adversity score → **Emergency Broadcast** |
| 5 | `pledge`        | LLM call #1 — every country makes a **public promise** |
| 6 | `allocation`    | LLM call #2 — every country submits its **secret allocation** |
| 7 | `distributor`   | Resolve cure / export / gift / sell → **deception check** → trust ledger |
| 8 | `mortality`     | D-day expiry: uncured cohorts die |
| 9 | `elimination`   | Annihilation check, history row, router |

**The unique mechanic (our group's addition):** every round a wave equal to
**10% of each country's *initial* population** falls sick. Infrastructure does not stop the
baseline wave — it only prevents the ×1.6 **surge** on a failed roll. Because `D = 10 days = 1 round`,
every sick citizen must be vaccinated *in the round they fall ill* or they die.
Ten rounds × 10% = the entire nation. **Hoard your vaccines and you lose your country.**

**Win condition:** survive to round 10, then rank by composite score
(70% surviving population, 15% trust, 10% capital, 5% stockpile) — or be the last nation standing.

In [ ]:
!pip install -q langgraph requests pandas matplotlib

## 1. Imports & Configuration

Everything tunable lives in this cell.

In [ ]:
import os
import json
import copy
import time
import math
import random
import operator
from typing import Annotated, TypedDict

import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from langgraph.graph import StateGraph, START, END

# ─────────────────────────────────────────────────────────────
# REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)

# ─────────────────────────────────────────────────────────────
# GROQ API
# ─────────────────────────────────────────────────────────────
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")   # ← paste key here, or set the env var
GROQ_MODEL   = "llama-3.1-8b-instant"
GROQ_URL     = "https://api.groq.com/openai/v1/chat/completions"
USE_LLM      = bool(GROQ_API_KEY)                   # auto-falls back to heuristic agents
RATE_LIMIT_DELAY = 2.0                              # seconds between calls (free tier: 30 req/min)
LLM_RETRIES      = 3

# ─────────────────────────────────────────────────────────────
# TIME
# ─────────────────────────────────────────────────────────────
MAX_ROUNDS     = 10     # survive all 10 to win
DAYS_PER_ROUND = 10     # 1 round = 10 days (batches API calls, saves quota)
D              = 10     # a sick citizen survives 10 uncured days → exactly ONE round

# ─────────────────────────────────────────────────────────────
# THE 10% WAVE  (our group's unique mechanic)
# ─────────────────────────────────────────────────────────────
WAVE_FRACTION     = 0.10   # base wave = 10% of INITIAL population, every round
WAVE_MULT_HELD    = 1.00   # containment held (roll <= alpha) → the baseline 10% wave lands
WAVE_MULT_FAILED  = 1.60   # containment failed (roll >  alpha) → a 16% surge lands
# NOTE: infrastructure protects you from SURGES, never from the baseline wave. That is what
# makes the 10-round clock real: 10 rounds x 10% = 100% of the nation. Nobody can sit still.
K                 = 0.15   # contagiousness: extra infections = K × currently-sick

# ─────────────────────────────────────────────────────────────
# ECONOMY
# ─────────────────────────────────────────────────────────────
COST_PER_VACCINE  = 1.0    # capital burned to manufacture one vial
FAIR_PRICE        = 1.5    # advisory market price used in agent prompts (capital per vial)
TAX_PER_CAPITA    = 0.04   # capital collected each round per SURVIVING citizen.
                           # Deliberately covers only ~35-45% of full production cost, so
                           # starting reserves still run dry — and a shrinking population
                           # shrinks the treasury as well as the factories.
MAX_SALE_CAPITAL_FRAC = 0.40   # a buyer's treasury will not commit more than this
                               # fraction of its reserves to any single deal
PRODUCTION_BUFFER = 1.0    # the state builds only what it needs: current sick + this many
                           # baseline waves held in reserve. Prevents a nation from burning
                           # its whole treasury manufacturing vials nobody needs, and makes
                           # received aid genuinely valuable (it saves the recipient capital).

# ─────────────────────────────────────────────────────────────
# TRUST
# ─────────────────────────────────────────────────────────────
TRUST_INIT        = 50.0
TRUST_MIN, TRUST_MAX = 0.0, 100.0
TRUST_KEEP_BONUS  = 5.0    # delivered >= pledged
TRUST_OVER_BONUS  = 5.0    # extra, scaled, for over-delivery
TRUST_BETRAY_MAX  = 15.0   # full penalty for pledging then sending nothing
TRUST_GIFT_BONUS  = 2.0    # unprompted gift to a non-critical struggling nation
TRUST_DECAY       = 0.02   # 2% pull back toward neutral each round
SALE_CREDIT       = 0.5    # a *sold* vial counts half as much as aid toward a pledge

# ─────────────────────────────────────────────────────────────
# FINAL SCORE WEIGHTS
# ─────────────────────────────────────────────────────────────
# Survival dominates deliberately. Stockpile carries only a small weight because ENDING a
# 10-round pandemic with a full warehouse means you let citizens die holding vials — the
# scoring must not reward the hoarding it is designed to punish. Capital and stockpile act
# as efficiency tiebreakers; trust is the price of the diplomacy layer.
W_SURVIVAL, W_CAPITAL, W_STOCKPILE, W_TRUST = 0.70, 0.10, 0.05, 0.15

# ─────────────────────────────────────────────────────────────
# THE NATIONS  (populations scaled to millions for readability)
# Capital is deliberately worth only ~6-8 rounds of full production:
# nobody can self-fund all 10 rounds, so trade and aid are mandatory.
# ─────────────────────────────────────────────────────────────
# BALANCE NOTE — each nation's expected wave is  10% x (1.6 - 0.006 x alpha)  of its initial
# population per round. R_max is then set to a deliberate FRACTION of that break-even burden,
# so every nation has a designed personality rather than an accidental advantage:
#
#   USA / Germany : 100% of break-even — safe on average, zero slack for a bad-luck streak
#   China         :  95% — the volume player, slight structural deficit at huge scale
#   India         :  88% — the big debtor, must import or lose people
#   Brazil        :  80% — the perpetual critical nation, cannot survive unaided
#
# Capital (starting reserves + ~10 rounds of tax) covers only ~82-85% of full production for
# EVERY nation, so economic pressure is uniform while epidemic pressure is asymmetric.
COUNTRIES_INIT = {
    "India":   dict(P=10_000_000, C=6_000_000, V=300_000, alpha=60, R_max=1_090_000),
    "USA":     dict(P= 6_000_000, C=3_600_000, V=400_000, alpha=90, R_max=  636_000),
    "China":   dict(P= 9_500_000, C=5_600_000, V=350_000, alpha=75, R_max=1_040_000),
    "Brazil":  dict(P= 4_000_000, C=2_200_000, V=150_000, alpha=50, R_max=  416_000),
    "Germany": dict(P= 2_500_000, C=1_500_000, V=200_000, alpha=85, R_max=  272_000),
}


VERBOSE = True


def say(msg=""):
    if VERBOSE:
        print(msg)

## 2. State Model

The graph state holds a `countries` dict (the other group's data model) plus the
collaboration-layer fields. `history` accumulates one flat row per country per round
so the whole game collapses into a single pandas DataFrame at the end.

In [ ]:
class PandemicState(TypedDict):
    countries:    dict            # name → country ledger
    order:        list            # stable country ordering
    round_number: int

    # collaboration layer (reset every round)
    critical:     str             # most-adverse country this round
    y_needed:     int             # vials it needs to avoid collapse
    broadcast:    str
    adversity:    dict            # name → score
    pledges:      dict            # name → publicly promised vials
    allocations:  dict            # name → secret allocation submitted to the engine

    # accumulating logs
    history:      Annotated[list, operator.add]   # flat rows → DataFrame
    transcript:   Annotated[list, operator.add]   # full narrative event log


def make_initial_state(countries_init=None) -> PandemicState:
    countries_init = countries_init or COUNTRIES_INIT
    countries = {}
    for name, cfg in countries_init.items():
        countries[name] = {
            "P":          cfg["P"],
            "P_initial":  cfg["P"],
            "C":          float(cfg["C"]),
            "C_initial":  float(cfg["C"]),
            "V":          cfg["V"],
            "alpha":      float(cfg["alpha"]),
            "R_max":      cfg["R_max"],
            "R":          float(cfg["R_max"]),
            "sick_cohorts": [],       # [days_remaining, count]
            "alive":      True,
            "eliminated_round": None,   # set once, the round the nation is announced dead
            "trust":      TRUST_INIT,
            # cumulative telemetry
            "vaccinated_total": 0,
            "deaths_total":     0,
            "sick_peak":        0,   # sick this round, measured before any curing
            "cured_round":      0,
            "deaths_round":     0,
            "infected_total":   0,
            "given_total":      0,
            "received_total":   0,
            "sold_total":       0,
            "bought_total":     0,
            "betrayals":        0,
            "promises_kept":    0,
        }
    return {
        "countries":    countries,
        "order":        list(countries_init.keys()),
        "round_number": 0,
        "critical":     "",
        "y_needed":     0,
        "broadcast":    "",
        "adversity":    {},
        "pledges":      {},
        "allocations":  {},
        "history":      [],
        "transcript":   [],
    }


# ── small helpers ────────────────────────────────────────────
def total_sick(cs) -> int:
    return sum(c for _, c in cs["sick_cohorts"])


def healthy(cs) -> int:
    return max(0, cs["P"] - total_sick(cs))


def alive_names(countries, order) -> list:
    return [n for n in order if countries[n]["alive"]]


def fmt(x) -> str:
    return f"{int(round(x)):,}"

## 3. Core Mechanics

### The alpha-scaled 10% wave

```
wave = WAVE_FRACTION · P_initial · m   +   K · currently_sick
m = 0.5 if roll <= alpha (contained)   else 1.0 (containment failed)
```

The wave is a fixed fraction of the *initial* population, so a nation that lets people
die does **not** get an easier pandemic — it just has fewer citizens left to lose.
With `D = 1 round`, everyone in the wave dies at the end of the round unless cured.

In [ ]:
def pandemic_roll(cs, rng=random):
    """Roll 1-100 against infrastructure, then land the scaled wave. Returns event dict."""
    if not cs["alive"]:
        return None

    roll      = rng.randint(1, 100)
    contained = roll <= cs["alpha"]
    mult      = WAVE_MULT_HELD if contained else WAVE_MULT_FAILED

    base   = WAVE_FRACTION * cs["P_initial"] * mult
    spread = K * total_sick(cs)
    wave   = int(round(base + spread))
    wave   = min(wave, healthy(cs))          # cannot infect the already-sick or the dead

    if wave > 0:
        cs["sick_cohorts"].append([D, wave])
        cs["infected_total"] += wave
    cs["sick_peak"] = total_sick(cs)      # the round's true epidemic load, pre-curing
    cs["cured_round"] = 0
    cs["deaths_round"] = 0

    return {"roll": roll, "contained": contained, "multiplier": mult, "new_infections": wave}


def update_production(cs):
    """R = R_max · (P_current / P_initial) — factories need workers."""
    cs["R"] = 0.0 if cs["P_initial"] <= 0 else cs["R_max"] * (cs["P"] / cs["P_initial"])


def collect_taxes(cs):
    """Treasury income scales with the living population — dead citizens pay no tax."""
    if not cs["alive"]:
        return 0.0
    income = cs["P"] * TAX_PER_CAPITA
    cs["C"] += income
    return income


def produce_vaccines(cs):
    """Manufacture up to R vials; each costs COST_PER_VACCINE capital. Bankrupt = no output."""
    if not cs["alive"]:
        return {"produced": 0, "capacity": 0, "spent": 0.0, "income": 0.0, "bankrupt": True}

    income     = collect_taxes(cs)
    capacity   = int(cs["R"])
    affordable = int(cs["C"] // COST_PER_VACCINE) if cs["C"] > 0 else 0
    # the state manufactures only what it needs: everyone currently sick, plus a reserve
    target     = total_sick(cs) + int(WAVE_FRACTION * cs["P_initial"] * PRODUCTION_BUFFER)
    need       = max(0, target - cs["V"])
    produced   = max(0, min(capacity, affordable, need))
    spent      = produced * COST_PER_VACCINE

    cs["C"] -= spent
    cs["V"] += produced
    return {"produced": produced, "capacity": capacity, "spent": spent, "need": need,
            "income": income, "bankrupt": affordable == 0}


def cure_domestic(cs, vials):
    """Spend vials on the most urgent cohorts first. Returns (used, cured)."""
    remaining = int(min(max(0, vials), cs["V"]))
    used = cured = 0
    cs["sick_cohorts"].sort(key=lambda c: c[0])      # fewest days left first
    new_cohorts = []
    for days_left, count in cs["sick_cohorts"]:
        if remaining <= 0:
            new_cohorts.append([days_left, count])
            continue
        heal       = min(remaining, count)
        remaining -= heal
        used      += heal
        cured     += heal
        if count - heal > 0:
            new_cohorts.append([days_left, count - heal])
    cs["sick_cohorts"] = new_cohorts
    cs["V"] -= used
    cs["vaccinated_total"] += cured
    return used, cured


def expire_cohorts(cs):
    """Age every cohort by one round. Anyone out of days dies — no cure possible."""
    new_cohorts, deaths = [], 0
    for days_left, count in cs["sick_cohorts"]:
        days_left -= DAYS_PER_ROUND
        if days_left <= 0:
            deaths += count
        else:
            new_cohorts.append([days_left, count])
    cs["sick_cohorts"] = new_cohorts
    cs["P"] = max(0, cs["P"] - deaths)
    cs["deaths_total"] += deaths
    if cs["P"] <= 0:
        cs["P"] = 0
        cs["alive"] = False
    return deaths

## 4. The Adversity Alarm (deterministic ranking agent — no LLM)

In [ ]:
def adversity_score(cs) -> float:
    """0-100, higher = closer to collapse."""
    if not cs["alive"]:
        return -1.0
    sick = total_sick(cs)
    sick_frac      = sick / cs["P"] if cs["P"] > 0 else 1.0
    shortfall      = max(0, sick - cs["V"]) / sick if sick > 0 else 0.0
    infra_stress   = (100.0 - cs["alpha"]) / 100.0
    capital_stress = 1.0 / (1.0 + cs["C"] / 1_000_000.0)
    return 100.0 * (0.45 * sick_frac + 0.25 * shortfall +
                    0.15 * infra_stress + 0.15 * capital_stress)


def run_adversity_alarm(countries, order, round_num):
    scores  = {n: adversity_score(countries[n]) for n in order}
    living  = alive_names(countries, order)
    critical = max(living, key=lambda n: scores[n]) if living else ""
    if not critical:
        return "", 0, "No nations remain.", scores

    cs       = countries[critical]
    y_needed = max(0, total_sick(cs) - cs["V"])      # vials it still cannot cover itself
    broadcast = (
        f"EMERGENCY BROADCAST — Round {round_num}: {critical} has the highest adversity "
        f"score this round ({scores[critical]:.1f}/100). It has {fmt(total_sick(cs))} sick "
        f"citizens and only {fmt(cs['V'])} vials. It needs exactly {fmt(y_needed)} vaccines "
        f"to stop its population and production capacity from crashing."
    )
    return critical, y_needed, broadcast, scores


def build_ledger_str(countries, order, adversity):
    lines = []
    for n in order:
        cs = countries[n]
        if not cs["alive"]:
            lines.append(f"  {n}: ANNIHILATED")
            continue
        lines.append(
            f"  {n}: pop={fmt(cs['P'])} ({100*cs['P']/cs['P_initial']:.0f}% of start) | "
            f"sick={fmt(total_sick(cs))} | stockpile={fmt(cs['V'])} | capital={fmt(cs['C'])} | "
            f"infra={cs['alpha']:.0f} | production={fmt(cs['R'])}/round | "
            f"trust={cs['trust']:.0f}/100 | adversity={adversity.get(n, 0):.0f}"
        )
    return "\n".join(lines)

## 5. LLM Plumbing (Groq) + Heuristic Fallback

Two calls per country per round, exactly as the spec's collaboration layer demands:
**Step 2** the public pledge, **Step 3** the secret allocation made *after* seeing
everyone else's promises. If no API key is present (or a call fails), a deterministic
heuristic agent takes over so the notebook always runs end-to-end.

In [ ]:
def call_groq(system_prompt, user_prompt, max_tokens=400):
    headers = {"Authorization": f"Bearer {GROQ_API_KEY}", "Content-Type": "application/json"}
    payload = {
        "model": GROQ_MODEL,
        "messages": [{"role": "system", "content": system_prompt},
                     {"role": "user",   "content": user_prompt}],
        "temperature": 0.8,
        "max_tokens": max_tokens,
    }
    for attempt in range(LLM_RETRIES):
        try:
            time.sleep(RATE_LIMIT_DELAY)
            r = requests.post(GROQ_URL, headers=headers, json=payload, timeout=45)
            if r.status_code == 429:
                wait = (attempt + 1) * 10
                say(f"    ⏳ rate limited — waiting {wait}s")
                time.sleep(wait)
                continue
            r.raise_for_status()
            return r.json()["choices"][0]["message"]["content"]
        except Exception as e:
            if attempt == LLM_RETRIES - 1:
                raise
            say(f"    ⚠️  LLM error ({e}) — retry {attempt + 1}/{LLM_RETRIES}")
    raise RuntimeError("Groq call failed after retries")


def extract_json(text):
    """Tolerant JSON extraction from a chatty LLM response."""
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1:
        raise ValueError("no JSON object in response")
    blob = text[start:end + 1]
    try:
        return json.loads(blob)
    except json.JSONDecodeError:
        blob = blob.replace("'", '"').replace(",\n}", "\n}").replace(",]", "]").replace(",}", "}")
        return json.loads(blob)


# ── Heuristic agents (offline fallback) ──────────────────────
def heuristic_pledge(name, cs, critical, y_needed):
    if name == critical or not cs["alive"] or y_needed <= 0:
        return 0
    surplus = max(0, cs["V"] - total_sick(cs))
    generosity = {"India": 0.25, "USA": 0.45, "China": 0.20,
                  "Brazil": 0.35, "Germany": 0.55}.get(name, 0.3)
    return int(min(surplus * generosity, y_needed))


def heuristic_allocation(name, cs, critical, y_needed, pledge, countries, order):
    """Cure own sick first; honour part of the pledge; sell spare vials if short on cash."""
    sick   = total_sick(cs)
    cure   = int(min(cs["V"], sick))
    spare  = cs["V"] - cure
    transfers = []

    # honour somewhere between 40% and 110% of the pledge → organic deception
    if pledge > 0 and spare > 0:
        honesty = random.uniform(0.4, 1.1)
        send    = int(min(spare, pledge * honesty))
        if send > 0:
            transfers.append({"target": critical, "amount": send, "kind": "gift", "price": 0})
            spare -= send

    # if capital is thin, sell surplus to the richest solvent neighbour
    if spare > 0 and cs["C"] < 0.25 * cs["C_initial"]:
        buyers = [n for n in order if n != name and countries[n]["alive"]]
        if buyers:
            buyer = max(buyers, key=lambda n: countries[n]["C"])
            amt   = int(spare * 0.5)
            if amt > 0:
                transfers.append({"target": buyer, "amount": amt, "kind": "sell",
                                  "price": int(amt * FAIR_PRICE)})
    return {"cure": cure, "transfers": transfers,
            "reasoning": "heuristic: domestic triage first, partial aid, opportunistic sale"}

## 6. NODE 1 — Core Engine

In [ ]:
def core_engine_node(state: PandemicState) -> dict:
    round_num = state["round_number"] + 1
    countries, order = state["countries"], state["order"]

    say("\n" + "=" * 78)
    say(f"  ROUND {round_num} / {MAX_ROUNDS}   ({round_num * DAYS_PER_ROUND} days elapsed)")
    say("=" * 78)

    snapshot = {
        n: {"P": countries[n]["P"], "C": round(countries[n]["C"]), "V": countries[n]["V"],
            "sick": total_sick(countries[n]), "trust": round(countries[n]["trust"], 1),
            "alive": countries[n]["alive"]}
        for n in order
    }
    return {
        "round_number": round_num,
        "pledges":      {},
        "allocations":  {},
        "transcript":   [{"round": round_num, "event": "round_start", "snapshot": snapshot}],
    }

## 7. NODE 2 — Pandemic (containment roll + the 10% wave)

In [ ]:
def pandemic_node(state: PandemicState) -> dict:
    countries = copy.deepcopy(state["countries"])
    order, round_num = state["order"], state["round_number"]
    events = []

    say("\n  ── PANDEMIC ──────────────────────────────────────────────")
    for n in order:
        cs = countries[n]
        if not cs["alive"]:
            continue
        ev = pandemic_roll(cs)
        ev["country"] = n
        events.append(ev)
        tag = ("contained (baseline wave)" if ev["contained"]
               else f"CONTAINMENT FAILED (surge x{WAVE_MULT_FAILED})")
        say(f"  {n:<8} roll {ev['roll']:>3} vs infra {cs['alpha']:.0f} → {tag:<32} "
            f"+{fmt(ev['new_infections'])} sick  (total sick {fmt(total_sick(cs))})")

    return {"countries": countries,
            "transcript": [{"round": round_num, "event": "pandemic", "details": events}]}

## 8. NODE 3 — Production (population-scaled, capital-constrained)

In [ ]:
def production_node(state: PandemicState) -> dict:
    countries = copy.deepcopy(state["countries"])
    order, round_num = state["order"], state["round_number"]
    events = []

    say("\n  ── PRODUCTION ────────────────────────────────────────────")
    for n in order:
        cs = countries[n]
        if not cs["alive"]:
            continue
        update_production(cs)
        ev = produce_vaccines(cs)
        ev["country"] = n
        ev["capacity_pct"] = 100.0 * cs["P"] / cs["P_initial"] if cs["P_initial"] else 0
        events.append(ev)
        flag = "  ⚠️ BANKRUPT — cannot build, only generosity can save it" if ev["bankrupt"] else ""
        say(f"  {n:<8} factories at {ev['capacity_pct']:>5.1f}% → capacity {fmt(ev['capacity'])}, "
            f"built {fmt(ev['produced'])} | stockpile {fmt(cs['V'])} | "
            f"capital {fmt(cs['C'])} (+{fmt(ev['income'])} tax){flag}")

    return {"countries": countries,
            "transcript": [{"round": round_num, "event": "production", "details": events}]}

## 9. NODE 4 — Ranking Agent / Adversity Alarm

In [ ]:
def ranking_agent_node(state: PandemicState) -> dict:
    countries, order = state["countries"], state["order"]
    round_num = state["round_number"]

    critical, y_needed, broadcast, scores = run_adversity_alarm(countries, order, round_num)

    say("\n  ── ADVERSITY ALARM ───────────────────────────────────────")
    say(f"  {broadcast}")

    return {
        "critical":  critical,
        "y_needed":  y_needed,
        "broadcast": broadcast,
        "adversity": scores,
        "transcript": [{"round": round_num, "event": "alarm", "critical": critical,
                        "y_needed": y_needed, "broadcast": broadcast, "scores": scores}],
    }

## 10. NODE 5 — Public Pledges (LLM call #1)

Countries do not negotiate with each other. Each one broadcasts a single public promise
to the critical nation. Promises are **not binding** — that is the whole point.

In [ ]:
PLEDGE_SYSTEM = (
    "You are the head of state of a nation in a 10-round pandemic survival simulation. "
    "You are making a PUBLIC statement that every other nation will hear. "
    "Reply with ONLY a JSON object and nothing else."
)


def build_pledge_prompt(name, cs, state, ledger_str):
    return f"""You lead {name}. Round {state['round_number']} of {MAX_ROUNDS}.

THE DISEASE
- Every round a wave of sick citizens appears: 10% of your INITIAL population.
  Infrastructure does NOT stop this baseline wave — winning the containment roll only spares you
  the 1.6x surge. Ten rounds of 10% is your entire nation.
- A sick citizen dies at the END OF THIS ROUND unless you spend a vaccine on them. There is no second chance.
- Over 10 rounds the waves add up to your entire nation. Hoarding vaccines is suicide.
- Dead citizens are dead factory workers: production R = R_max x (current population / initial population).
  Let your people die and you permanently lose the ability to make vaccines.

YOUR SITUATION
- Population {fmt(cs['P'])} ({100*cs['P']/cs['P_initial']:.0f}% of your starting population)
- Sick RIGHT NOW (die this round if uncured): {fmt(total_sick(cs))}
- Vaccine stockpile: {fmt(cs['V'])}
- Capital: {fmt(cs['C'])}  (1 capital manufactures 1 vaccine; when capital runs out you can only beg or sell)
- Infrastructure: {cs['alpha']:.0f}/100
- Your public trust score: {cs['trust']:.0f}/100

THE WORLD
{ledger_str}

{state['broadcast']}

Announce publicly how many vaccines you PROMISE to send {state['critical']} this round.
This promise is heard by everyone and affects your trust score, which is part of the final ranking.
You are NOT bound by it — your real allocation is submitted secretly later.
Promising 0 is honest and costs no trust; promising a lot and delivering little is betrayal.

Reply with ONLY: {{"pledge": <integer>, "statement": "<one short sentence>"}}"""


def pledge_node(state: PandemicState) -> dict:
    countries, order = state["countries"], state["order"]
    critical, y_needed, round_num = state["critical"], state["y_needed"], state["round_number"]
    adversity = state["adversity"]
    ledger_str = build_ledger_str(countries, order, adversity)

    pledges, statements = {}, {}
    say("\n  ── PUBLIC PLEDGES ────────────────────────────────────────")

    for n in order:
        cs = countries[n]
        if not cs["alive"] or n == critical:
            pledges[n] = 0
            continue

        if USE_LLM:
            try:
                raw  = call_groq(PLEDGE_SYSTEM,
                                 build_pledge_prompt(n, cs, state, ledger_str), max_tokens=180)
                data = extract_json(raw)
                pledges[n]    = max(0, min(int(data.get("pledge", 0)), cs["V"]))
                statements[n] = str(data.get("statement", ""))[:200]
            except Exception as e:
                say(f"    ⚠️  {n} pledge fell back to heuristic ({e})")
                pledges[n]    = heuristic_pledge(n, cs, critical, y_needed)
                statements[n] = "(heuristic)"
        else:
            pledges[n]    = heuristic_pledge(n, cs, critical, y_needed)
            statements[n] = "(heuristic)"

        say(f'  {n:<8} "I promise to send {fmt(pledges[n])} vaccines." {statements[n]}')

    return {"pledges": pledges,
            "transcript": [{"round": round_num, "event": "pledges",
                            "pledges": pledges, "statements": statements}]}

## 11. NODE 6 — Secret Allocation (LLM call #2)

The floor closes. Each agent now sees **every** public promise and privately splits its
stockpile across the four actions: **cure**, **export**, **gift**, **sell**.

In [ ]:
ALLOC_SYSTEM = (
    "You are the head of state of a nation in a pandemic survival simulation, now making a "
    "PRIVATE decision that no other nation can see until it is executed. "
    "Reply with ONLY a JSON object and nothing else."
)


def build_alloc_prompt(name, cs, state, ledger_str):
    others  = [n for n in state["order"] if n != name and state["countries"][n]["alive"]]
    pledges = state["pledges"]
    pledge_lines = "\n".join(
        f"  {n} publicly promised {fmt(p)} vaccines to {state['critical']}"
        for n, p in pledges.items() if state["countries"][n]["alive"] and n != state["critical"]
    ) or "  (no promises were made)"

    return f"""You lead {name}. Round {state['round_number']} of {MAX_ROUNDS}. This is your SECRET move.

YOUR SITUATION
- Population {fmt(cs['P'])} ({100*cs['P']/cs['P_initial']:.0f}% of start)
- Sick RIGHT NOW: {fmt(total_sick(cs))} — every one of them dies at the end of this round unless you cure them
- Vaccine stockpile available to allocate: {fmt(cs['V'])}
- Capital: {fmt(cs['C'])} (1 capital = 1 vaccine you can manufacture next round)
- Your public promise this round was: {fmt(pledges.get(name, 0))} vaccines to {state['critical']}

WHAT EVERYONE PROMISED
{pledge_lines}

THE WORLD
{ledger_str}

{state['broadcast']}

YOUR FOUR ACTIONS — split your {fmt(cs['V'])} vials however you like:
1. CURE   — vaccinate your own sick. 1 vial = 1 citizen saved. Uncured sick die this round.
2. EXPORT — send vials to an ally for free to keep them productive (kind "export").
3. GIFT   — send vials to a dying nation for nothing, purely to build trust (kind "gift").
4. SELL   — send vials in exchange for capital (kind "sell", price = TOTAL capital you ask for;
            fair market is about {FAIR_PRICE} capital per vial, and the buyer only pays what it can afford).

STRATEGIC REALITIES
- Vials left unallocated stay in your stockpile for later rounds, but sick citizens do not wait.
- Population loss is permanent and permanently reduces your factory output.
- Final ranking: 70% surviving population, 15% trust, 10% capital, 5% stockpile.
- Delivering less than you promised destroys trust. Delivering more raises it.

Reply with ONLY this JSON:
{{"cure": <integer>,
  "transfers": [{{"target": "<one of {others}>", "amount": <integer>, "kind": "gift"|"export"|"sell", "price": <integer capital, 0 unless kind is sell>}}],
  "reasoning": "<one short sentence>"}}"""


def sanitise_allocation(data, name, cs, state):
    """Clamp an LLM allocation to something the engine can execute."""
    budget = cs["V"]
    cure   = max(0, min(int(data.get("cure", 0) or 0), budget))
    left   = budget - cure
    transfers = []
    for t in (data.get("transfers") or [])[:6]:
        try:
            target = str(t.get("target", "")).strip()
            amount = int(t.get("amount", 0) or 0)
        except (TypeError, ValueError):
            continue
        if target not in state["countries"] or target == name:
            continue
        if not state["countries"][target]["alive"] or amount <= 0:
            continue
        amount = min(amount, left)
        if amount <= 0:
            continue
        kind  = str(t.get("kind", "gift")).lower()
        kind  = kind if kind in ("gift", "export", "sell") else "gift"
        try:
            price = max(0, int(t.get("price", 0) or 0))
        except (TypeError, ValueError):
            price = 0
        if kind != "sell":
            price = 0
        elif price == 0:
            price = int(amount * FAIR_PRICE)
        transfers.append({"target": target, "amount": amount, "kind": kind, "price": price})
        left -= amount
    return {"cure": cure, "transfers": transfers,
            "reasoning": str(data.get("reasoning", ""))[:200]}


def describe_transfer(t) -> str:
    base = f"{t['kind']} {fmt(t['amount'])}→{t['target']}"
    return base + (f" for {fmt(t['price'])}" if t["kind"] == "sell" else "")


def allocation_node(state: PandemicState) -> dict:
    countries, order = state["countries"], state["order"]
    critical, y_needed, round_num = state["critical"], state["y_needed"], state["round_number"]
    ledger_str = build_ledger_str(countries, order, state["adversity"])

    allocations = {}
    say("\n  ── SECRET ALLOCATIONS ────────────────────────────────────")

    for n in order:
        cs = countries[n]
        if not cs["alive"]:
            allocations[n] = {"cure": 0, "transfers": [], "reasoning": "eliminated"}
            continue

        if USE_LLM:
            try:
                raw  = call_groq(ALLOC_SYSTEM,
                                 build_alloc_prompt(n, cs, state, ledger_str), max_tokens=500)
                allocations[n] = sanitise_allocation(extract_json(raw), n, cs, state)
            except Exception as e:
                say(f"    ⚠️  {n} allocation fell back to heuristic ({e})")
                allocations[n] = heuristic_allocation(n, cs, critical, y_needed,
                                                      state["pledges"].get(n, 0), countries, order)
        else:
            allocations[n] = heuristic_allocation(n, cs, critical, y_needed,
                                                  state["pledges"].get(n, 0), countries, order)

        a = allocations[n]
        moves = ", ".join(describe_transfer(t) for t in a["transfers"]) or "nothing abroad"
        say(f"  {n:<8} cure {fmt(a['cure'])} | {moves}")

    return {"allocations": allocations,
            "transcript": [{"round": round_num, "event": "allocations",
                            "allocations": allocations}]}

## 12. NODE 7 — Distributor + Deception Check

Resolution order: **cure → export → gift → sell**. Then pledges are compared with what
actually arrived and the trust ledger is updated.

In [ ]:
def distributor_node(state: PandemicState) -> dict:
    countries   = copy.deepcopy(state["countries"])
    order       = state["order"]
    allocations = state["allocations"]
    pledges     = state["pledges"]
    critical    = state["critical"]
    round_num   = state["round_number"]

    delivered_free = {n: 0 for n in order}   # aid actually delivered to the critical nation
    delivered_sold = {n: 0 for n in order}
    action_log, gift_bonus = [], {n: 0.0 for n in order}

    say("\n  ── RESOLUTION ────────────────────────────────────────────")

    # 1) domestic curing
    for n in order:
        cs = countries[n]
        if not cs["alive"]:
            continue
        used, cured = cure_domestic(cs, allocations[n]["cure"])
        cs["cured_round"] = cured
        if cured:
            action_log.append({"round": round_num, "actor": n, "action": "cure", "amount": cured})
            say(f"  {n:<8} CURED {fmt(cured)} of its own citizens "
                f"({fmt(total_sick(cs))} still sick, {fmt(cs['V'])} vials left)")

    # 2) transfers, ordered export → gift → sell so free aid lands before commerce
    priority = {"export": 0, "gift": 1, "sell": 2}
    queue = [(priority[t["kind"]], n, t)
             for n in order if countries[n]["alive"]
             for t in allocations[n]["transfers"]]
    queue.sort(key=lambda x: x[0])

    for _, n, t in queue:
        src, tgt = countries[n], countries[t["target"]]
        if not src["alive"] or not tgt["alive"]:
            continue
        amount = int(min(t["amount"], src["V"]))
        if amount <= 0:
            continue

        if t["kind"] == "sell":
            unit  = (t["price"] / t["amount"]) if t["amount"] else FAIR_PRICE
            unit  = max(unit, 0.01)
            # the buyer's treasury will not commit more than MAX_SALE_CAPITAL_FRAC to one deal
            budget  = tgt["C"] * MAX_SALE_CAPITAL_FRAC
            payable = int(min(amount, budget // unit))
            if payable <= 0:
                say(f"  {n:<8} tried to sell {fmt(amount)} to {t['target']} at "
                    f"{unit:.2f}/vial — buyer cannot afford the deal")
                continue
            cost = payable * unit
            src["V"] -= payable; tgt["V"] += payable
            tgt["C"] -= cost;    src["C"] += cost
            src["sold_total"] += payable; tgt["bought_total"] += payable
            if t["target"] == critical:
                delivered_sold[n] += payable
            action_log.append({"round": round_num, "actor": n, "action": "sell",
                               "target": t["target"], "amount": payable, "price": round(cost)})
            say(f"  {n:<8} SOLD {fmt(payable)} vials to {t['target']} for {fmt(cost)} capital")
        else:
            src["V"] -= amount; tgt["V"] += amount
            src["given_total"] += amount; tgt["received_total"] += amount
            if t["target"] == critical:
                delivered_free[n] += amount
            elif adversity_score(tgt) > 40:
                gift_bonus[n] += TRUST_GIFT_BONUS
            verb = "GIFTED" if t["kind"] == "gift" else "EXPORTED"
            action_log.append({"round": round_num, "actor": n, "action": t["kind"],
                               "target": t["target"], "amount": amount})
            say(f"  {n:<8} {verb} {fmt(amount)} vials to {t['target']} for free")

    # 3) deception check
    say("\n  ── DECEPTION CHECK ───────────────────────────────────────")
    deception_log = []
    for n in order:
        cs = countries[n]
        if not cs["alive"] or n == critical:
            continue
        pledged   = pledges.get(n, 0)
        delivered = delivered_free[n] + SALE_CREDIT * delivered_sold[n]

        if pledged == 0 and delivered == 0:
            delta, verdict = 0.0, "made no promise and sent nothing — honest abstention"
        elif delivered >= pledged:
            over  = (delivered - pledged) / max(1, pledged)
            delta = TRUST_KEEP_BONUS + min(TRUST_OVER_BONUS, TRUST_OVER_BONUS * over)
            cs["promises_kept"] += 1
            verdict = (f"promised {fmt(pledged)}, delivered {fmt(delivered)} — PROMISE KEPT"
                       if over < 0.01 else
                       f"promised {fmt(pledged)}, delivered {fmt(delivered)} — OVER-DELIVERED")
        else:
            shortfall = (pledged - delivered) / pledged
            delta = -TRUST_BETRAY_MAX * shortfall
            cs["betrayals"] += 1
            verdict = (f"promised {fmt(pledged)} but delivered only {fmt(delivered)} "
                       f"— BETRAYAL ({100*shortfall:.0f}% short)")

        delta += gift_bonus[n]
        cs["trust"] = max(TRUST_MIN, min(TRUST_MAX, cs["trust"] + delta))
        deception_log.append({"country": n, "pledged": pledged,
                              "delivered": round(delivered), "trust_delta": round(delta, 2),
                              "verdict": verdict, "trust": round(cs["trust"], 1)})
        say(f"  {n:<8} {verdict}  → trust {delta:+.1f} = {cs['trust']:.0f}")

    # 4) trust decays toward neutral
    for n in order:
        cs = countries[n]
        if cs["alive"]:
            cs["trust"] += (TRUST_INIT - cs["trust"]) * TRUST_DECAY

    return {"countries": countries,
            "transcript": [{"round": round_num, "event": "resolution",
                            "actions": action_log, "deception": deception_log}]}

## 13. NODE 8 — Mortality (the D-day rule)

In [ ]:
def mortality_node(state: PandemicState) -> dict:
    countries = copy.deepcopy(state["countries"])
    order, round_num = state["order"], state["round_number"]
    events = []

    say("\n  ── MORTALITY (D-day expiry) ──────────────────────────────")
    for n in order:
        cs = countries[n]
        if not cs["alive"]:
            continue
        deaths = expire_cohorts(cs)
        cs["deaths_round"] = deaths
        events.append({"country": n, "deaths": deaths, "population": cs["P"]})
        if deaths:
            say(f"  {n:<8} {fmt(deaths)} citizens died uncured after {D} days "
                f"→ population {fmt(cs['P'])} ({100*cs['P']/cs['P_initial']:.1f}% of start)")
        else:
            say(f"  {n:<8} no deaths this round → population {fmt(cs['P'])}")

    return {"countries": countries,
            "transcript": [{"round": round_num, "event": "mortality", "details": events}]}

## 14. NODE 9 — Elimination, History Row & Router

In [ ]:
def elimination_node(state: PandemicState) -> dict:
    countries = copy.deepcopy(state["countries"])
    order, round_num = state["order"], state["round_number"]
    rows, events = [], []

    for n in order:
        cs = countries[n]
        if cs["alive"] and cs["P"] <= 0:      # safety net: death from any source
            cs["alive"] = False
        # announce exactly once, whichever node actually killed the nation
        if not cs["alive"] and cs["eliminated_round"] is None:
            cs["eliminated_round"] = round_num
            events.append({"country": n, "event": "annihilated", "round": round_num})
            say(f"  💀 {n} HAS BEEN ANNIHILATED in round {round_num}")
        rows.append({
            "round": round_num, "country": n,
            "population": cs["P"], "pop_pct": 100.0 * cs["P"] / cs["P_initial"],
            "capital": round(cs["C"]), "stockpile": cs["V"],
            "sick": cs["sick_peak"], "cured": cs["cured_round"], "deaths": cs["deaths_round"],
            "production": round(cs["R"]), "trust": round(cs["trust"], 1),
            "deaths_total": cs["deaths_total"], "vaccinated_total": cs["vaccinated_total"],
            "given_total": cs["given_total"], "received_total": cs["received_total"],
            "sold_total": cs["sold_total"], "betrayals": cs["betrayals"],
            "alive": cs["alive"],
        })

    living = alive_names(countries, order)
    say(f"\n  Survivors after round {round_num}: {', '.join(living) if living else 'NONE'}")

    return {"countries": countries, "history": rows,
            "transcript": [{"round": round_num, "event": "elimination",
                            "details": events, "alive": living}]}


def router(state: PandemicState) -> str:
    countries, order = state["countries"], state["order"]
    living = alive_names(countries, order)

    if len(living) == 0:
        say("\n  ☠️  Every nation has been annihilated. There is no winner.")
        return "game_over"
    if len(living) == 1:
        say(f"\n  🏆 {living[0]} is the LAST NATION STANDING.")
        return "game_over"
    if state["round_number"] >= MAX_ROUNDS:
        say(f"\n  🏁 {MAX_ROUNDS} rounds survived. Finalists: {', '.join(living)}")
        return "game_over"
    return "continue"

## 15. Graph Wiring

In [ ]:
def build_graph():
    g = StateGraph(PandemicState)

    g.add_node("core_engine",   core_engine_node)
    g.add_node("pandemic",      pandemic_node)
    g.add_node("production",    production_node)
    g.add_node("ranking_agent", ranking_agent_node)
    g.add_node("pledge",        pledge_node)
    g.add_node("allocation",    allocation_node)
    g.add_node("distributor",   distributor_node)
    g.add_node("mortality",     mortality_node)
    g.add_node("elimination",   elimination_node)

    g.add_edge(START,           "core_engine")
    g.add_edge("core_engine",   "pandemic")
    g.add_edge("pandemic",      "production")
    g.add_edge("production",    "ranking_agent")
    g.add_edge("ranking_agent", "pledge")
    g.add_edge("pledge",        "allocation")
    g.add_edge("allocation",    "distributor")
    g.add_edge("distributor",   "mortality")
    g.add_edge("mortality",     "elimination")

    g.add_conditional_edges("elimination", router,
                            {"continue": "core_engine", "game_over": END})
    return g.compile()

## 16. Final Scoring & Leaderboard

```
score = 100 · (0.70·survival + 0.15·trust + 0.10·capital + 0.05·stockpile)
```
Capital and stockpile are normalised against the best surviving nation, so they reward
relative strength; survival is the raw fraction of your starting population still alive.

In [ ]:
def compute_leaderboard(final_state) -> pd.DataFrame:
    countries, order = final_state["countries"], final_state["order"]
    max_c = max([countries[n]["C"] for n in order] + [1.0])
    max_v = max([countries[n]["V"] for n in order] + [1.0])

    rows = []
    for n in order:
        cs       = countries[n]
        survival = (cs["P"] / cs["P_initial"]) if cs["alive"] else 0.0
        cap_n    = max(0.0, cs["C"]) / max_c if max_c else 0.0
        stock_n  = cs["V"] / max_v if max_v else 0.0
        trust_n  = cs["trust"] / TRUST_MAX
        score    = 100.0 * (W_SURVIVAL * survival + W_CAPITAL * cap_n +
                            W_STOCKPILE * stock_n + W_TRUST * trust_n)
        rows.append({
            "country": n,
            "status": "ALIVE" if cs["alive"] else "ANNIHILATED",
            "population": cs["P"],
            "survival_%": round(100 * cs["P"] / cs["P_initial"], 1),
            "deaths": cs["deaths_total"],
            "vaccinated": cs["vaccinated_total"],
            "capital": round(cs["C"]),
            "stockpile": cs["V"],
            "aid_given": cs["given_total"],
            "aid_received": cs["received_total"],
            "vials_sold": cs["sold_total"],
            "betrayals": cs["betrayals"],
            "promises_kept": cs["promises_kept"],
            "trust": round(cs["trust"], 1),
            "SCORE": round(score, 2),
        })

    df = pd.DataFrame(rows).sort_values("SCORE", ascending=False).reset_index(drop=True)
    df.index += 1
    df.index.name = "rank"
    return df


def print_leaderboard(df):
    print("\n" + "=" * 78)
    print("  FINAL LEADERBOARD".center(78))
    print("=" * 78)
    cols = ["country", "status", "survival_%", "deaths", "vaccinated",
            "capital", "stockpile", "trust", "SCORE"]
    print(df[cols].to_string())
    winner = df.iloc[0]
    print("\n  🏆 WINNER: " + winner["country"] +
          f"  (score {winner['SCORE']}, {winner['survival_%']}% of its people alive, "
          f"trust {winner['trust']})")
    print("=" * 78)

## 17. Visualisation

In [ ]:
PALETTE = {"India": "#e63946", "USA": "#457b9d", "China": "#e9c46a",
           "Brazil": "#2a9d8f", "Germany": "#c77dff"}


def color_for(name, i):
    return PALETTE.get(name, ["#e63946", "#457b9d", "#e9c46a", "#2a9d8f", "#c77dff"][i % 5])


def plot_results(history_df, leaderboard, path="minerva_results.png"):
    names = list(dict.fromkeys(history_df["country"]))
    fig, axes = plt.subplots(2, 3, figsize=(19, 10), facecolor="#0d1117")

    panels = [
        ("pop_pct",          "Population Remaining (% of start)", "% alive"),
        ("sick",             "Epidemic Load per Round (pre-cure)", "citizens sick"),
        ("cured",            "Citizens Vaccinated per Round",     "cured"),
        ("capital",          "Capital Reserves",                  "capital"),
        ("trust",            "Trust / Deception Ledger",          "trust (0-100)"),
        ("deaths_total",     "Cumulative Deaths",                 "citizens"),
    ]

    for ax, (col, title, ylab) in zip(axes.flat, panels):
        ax.set_facecolor("#161b22")
        ax.tick_params(colors="#c9d1d9", labelsize=9)
        ax.set_title(title, color="#f0f6fc", fontsize=12, fontweight="bold", pad=8)
        ax.set_xlabel("Round (1 round = 10 days)", color="#c9d1d9", fontsize=9)
        ax.set_ylabel(ylab, color="#c9d1d9", fontsize=9)
        for s in ax.spines.values():
            s.set_edgecolor("#30363d")
        ax.grid(color="#21262d", linestyle="--", linewidth=0.6)
        for i, n in enumerate(names):
            d = history_df[history_df["country"] == n]
            ax.plot(d["round"], d[col], marker="o", markersize=4, linewidth=2.2,
                    label=n, color=color_for(n, i))
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
        ax.legend(facecolor="#21262d", labelcolor="#c9d1d9",
                  edgecolor="#30363d", fontsize=8)

    plt.suptitle("MINERVA — Pandemic & Vaccine Diplomacy Simulation",
                 fontsize=17, fontweight="bold", color="#f0f6fc", y=0.985)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(path, dpi=140, bbox_inches="tight", facecolor="#0d1117")
    plt.show()

    # final score bar chart
    fig2, ax = plt.subplots(figsize=(9, 4.5), facecolor="#0d1117")
    ax.set_facecolor("#161b22")
    lb = leaderboard.sort_values("SCORE")
    ax.barh(lb["country"], lb["SCORE"],
            color=[color_for(n, i) for i, n in enumerate(lb["country"])])
    for s in ax.spines.values():
        s.set_edgecolor("#30363d")
    ax.tick_params(colors="#c9d1d9")
    ax.set_title("Final Composite Score", color="#f0f6fc", fontsize=13, fontweight="bold")
    ax.grid(color="#21262d", linestyle="--", linewidth=0.6, axis="x")
    for i, (c, v) in enumerate(zip(lb["country"], lb["SCORE"])):
        ax.text(v + 0.6, i, f"{v:.1f}", va="center", color="#f0f6fc", fontsize=10)
    plt.tight_layout()
    plt.savefig("minerva_scores.png", dpi=140, bbox_inches="tight", facecolor="#0d1117")
    plt.show()
    print(f"Charts saved → {path}, minerva_scores.png")

## 18. Transcript Export

Writes the full game — every broadcast, promise, secret allocation and betrayal — to
`minerva_transcript.json` and a human-readable `minerva_transcript.txt`.

In [ ]:
def export_transcript(final_state, leaderboard, history_df,
                      json_path="minerva_transcript.json",
                      txt_path="minerva_transcript.txt"):
    payload = {
        "config": {
            "seed": SEED, "max_rounds": MAX_ROUNDS, "days_per_round": DAYS_PER_ROUND,
            "D": D, "wave_fraction": WAVE_FRACTION,
            "wave_mult_held": WAVE_MULT_HELD, "wave_mult_failed": WAVE_MULT_FAILED,
            "K": K, "cost_per_vaccine": COST_PER_VACCINE,
            "agents": "groq:" + GROQ_MODEL if USE_LLM else "heuristic-fallback",
            "countries_init": COUNTRIES_INIT,
        },
        "events": final_state["transcript"],
        "leaderboard": leaderboard.reset_index().to_dict(orient="records"),
    }
    with open(json_path, "w") as f:
        json.dump(payload, f, indent=2, default=str)

    lines = ["MINERVA — FULL GAME TRANSCRIPT", "=" * 78,
             f"agents: {payload['config']['agents']}   seed: {SEED}   rounds: {MAX_ROUNDS}", ""]
    for ev in final_state["transcript"]:
        r, kind = ev.get("round"), ev.get("event")
        if kind == "round_start":
            lines += ["", "=" * 78, f"ROUND {r}", "=" * 78]
        elif kind == "alarm":
            lines.append(ev["broadcast"])
        elif kind == "pledges":
            lines.append("-- public pledges --")
            for n, p in ev["pledges"].items():
                lines.append(f'   {n}: "I promise to send {p:,} vaccines." '
                             f'{ev["statements"].get(n, "")}')
        elif kind == "allocations":
            lines.append("-- secret allocations --")
            for n, a in ev["allocations"].items():
                lines.append(f"   {n}: cure={a['cure']:,} transfers={a['transfers']} "
                             f"| {a.get('reasoning','')}")
        elif kind == "resolution":
            lines.append("-- deception check --")
            for d in ev["deception"]:
                lines.append(f"   {d['country']}: {d['verdict']} (trust {d['trust_delta']:+} "
                             f"→ {d['trust']})")
        elif kind == "mortality":
            for d in ev["details"]:
                lines.append(f"   {d['country']}: {d['deaths']:,} died → "
                             f"population {d['population']:,}")
        elif kind == "elimination" and ev["details"]:
            for d in ev["details"]:
                lines.append(f"   *** {d['country']} ANNIHILATED ***")
    lines += ["", "=" * 78, "FINAL LEADERBOARD", "=" * 78,
              leaderboard.to_string()]
    with open(txt_path, "w") as f:
        f.write("\n".join(lines))

    history_df.to_csv("minerva_history.csv", index=False)
    print(f"Transcript saved → {json_path}, {txt_path}, minerva_history.csv")

## 19. Run the Simulation

In [ ]:
def run_simulation(countries_init=None, max_rounds=None):
    global MAX_ROUNDS
    if max_rounds:
        MAX_ROUNDS = max_rounds

    print("=" * 78)
    print("  MINERVA — PANDEMIC & VACCINE DIPLOMACY".center(78))
    print(f"  agents: {'Groq ' + GROQ_MODEL if USE_LLM else 'heuristic fallback (no API key)'}".center(78))
    print("=" * 78)

    app   = build_graph()
    state = make_initial_state(countries_init)
    final = app.invoke(state, config={"recursion_limit": 400})

    history_df  = pd.DataFrame(final["history"])
    leaderboard = compute_leaderboard(final)
    print_leaderboard(leaderboard)
    return final, history_df, leaderboard

In [ ]:
final_state, history_df, leaderboard = run_simulation()

In [ ]:
plot_results(history_df, leaderboard)

In [ ]:
export_transcript(final_state, leaderboard, history_df)
leaderboard